# Exploratory Data Analysis (EDA)

## Table of Contents
1. [Dataset Overview](#dataset-overview)
2. [Handling Missing Values](#handling-missing-values)
3. [Feature Distributions](#feature-distributions)
4. [Possible Biases](#possible-biases)
5. [Correlations](#correlations)


. [Correlations](#correlations)


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## Dataset Overview

[Provide a high-level overview of the dataset. This should include the source of the dataset, the number of samples, the number of features, and example showing the structure of the dataset.]


In [ ]:
import pandas as pd

# Load the data

!wget -O hmp2_metadata.csv "https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/metadata/hmp2_metadata_2018-08-20.csv" 

# Replace 'your_dataset.csv' with the path to your actual dataset
df = pd.read_csv('hmp2_metadata.csv')

# Number of samples
num_samples = df.shape[0]

# Number of features
num_features = df.shape[1]

# Display these dataset characteristics
print(f"Number of samples: {num_samples}")
print(f"Number of features: {num_features}")

# Display the first few rows of the dataframe to show the structure
print("Example data:")
print(df.head())

# ============================================
# PART 1: DATA EXPLORATION
# ============================================

# Display the distribution of data types in the dataset
print("=== DATA TYPES IN STUDY ===")
print(df['data_type'].value_counts())
print(f"\nTotal rows: {df.shape[0]}, Total columns: {df.shape[1]}")
print(f"Unique participants: {df['Participant ID'].nunique()}")

# Set display options to show ALL columns
pd.set_option('display.max_columns', None)  # None = unlimited
pd.set_option('display.max_rows', 100)      # Show first 100 rows
pd.set_option('display.width', None)        # Auto-detect width
pd.set_option('display.max_colwidth', None) # Show full column content

####
# Display the entire dataset to show the structure
# from IPython.display import display, HTML
# Convert to HTML with horizontal scroll
# df_html = df.to_html(escape=False)
# display(HTML(f"""
# <style>
# table.dataframe {{
#     overflow-x: auto;
#     display: block;
#     white-space: nowrap;
# }}
# </style>
# {df_html}
# """))

## Handling Missing Values

Missing columns: 
- Demographics
- Sample Type
- Medications
- Therapy durations
- Montreal Classification (different criteria for UC and CD patients)
- Disease Activity 

Approach: 
- Work on 90+ features that have all entries first
- Start with 2 subtypes with fewer features 

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_values


In [ ]:
# Handling missing values
# Example: Replacing NaN values with the mean value of the column
# df.fillna(df.mean(), inplace=True)

# Your code for handling missing values goes here
# ============================================
# PART 2: MISSING VALUES ANALYSIS
# ============================================

print("\n" + "="*60)
print("MISSING VALUES ANALYSIS")
print("="*60)

# Calculate missing values
missing_per_column = df.isnull().sum()
missing_percentage = (missing_per_column / len(df)) * 100

# Create summary DataFrame
missing_summary = pd.DataFrame({
    'Missing_Count': missing_per_column,
    'Missing_Percentage': missing_percentage,
    'Present_Percentage': 100 - missing_percentage
}).sort_values('Missing_Percentage', ascending=True)

# Show ALL columns with missing values
cols_with_missing = missing_summary[missing_summary['Missing_Count'] > 0]
print(f"\n📊 Columns WITH missing values: {len(cols_with_missing)} / {len(df.columns)}")
print(f"📊 Columns WITHOUT missing values: {len(df.columns) - len(cols_with_missing)} / {len(df.columns)}")

# Show top 50 columns with most missing data
print("\n" + "="*60)
print("TOP 50 COLUMNS WITH MOST MISSING VALUES")
print("="*60)
print(cols_with_missing.tail(50).to_string())

# Overall missing percentage
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]
print(f"\n📊 Overall missing data: {total_missing:,} cells ({(total_missing/total_cells)*100:.2f}%)")


# ============================================
# PART 3: KEY CLINICAL VARIABLES FOR IBD THERAPY PREDICTION
# ============================================

print("\n" + "="*60)
print("KEY CLINICAL VARIABLES FOR IBD THERAPY PREDICTION")
print("="*60)

# Define key clinical columns 
key_columns = {
    'Diagnosis': ['diagnosis', 'Research Project'],
    'Montreal Classification': ['Age at diagnosis (A)', 'baseline_montreal_location', 'Location (L) prior to first surgery', 'Behavior (B)', 'Extent (E)'],
    'Disease Activity': ['hbi', 'sccai', 'fecalcal', 'fecalcal_ng_ml', 'CRP (mg/L)', 'ESR (mm/hr)', 'SES-CD Score', "Modified Baron's Score"],
    'Demographics': ['sex', 'BMI', 'Height', 'Weight', 'race', 'Hispanic or Latino Origin'],
    'Medications': [ 'Antibiotics', 'Chemotherapy', 'Immunosuppressants (e.g. oral corticosteroids)','Lomotil', 'Dipentum (olsalazine)', 'Rowasa enemas (mesalamine enemas)', 'Canasa suppositories (mesalamine suppositories)', 'Flagyl (Metronidazole)', 'Cipro (Ciprofloxin)', 'Xifaxin (rifaxamin)', 'Levaquin', 'Other Antibiotic:', 'Prednisone', 'Entocort (Budesonide)', 'Imodium', 'Solumedrol (Medrol)','IV steroids','Cortenemas, Cortifoam, Proctofoam','Azathioprine (Imuran, Azasan)','Methotrexate','Mercaptopurine (Purinethol, 6MP)','VSL #3','FOS','Remicade (Infliximab)','Humira (Adalimumab)', 'DTO', 'Cimzia (Certlizumab)','Tysabri (Natalizumab)','Asacol (mesalamine)','Pentasa (mesalamine)','Lialda (mesalamine)','Apriso (mesalamine)','Colozal (balasalizide)','Sulfasalizine (Azulfidine)'],
    'Therapy Duration': ['Duration of Remicade use (months):', 'Duration of Humira use (months):', 'Duration of Cimzia use (months):', 'Duration of Tysabri use (months):'],
    'Response Indicators': ['is_inflamed', 'Disease_course', 'IN THE PAST SIX MONTHS, my disease has been:', '5a. How would you rate your health today with regard to your', '5b. How would you rate your health today with regard to you'],
    'Sample Type': ['data_type', 'stool_id', 'biopsy_location', 'Location']
}

print("\nChecking availability and missingness of key clinical variables:\n")
for category, columns in key_columns.items():
    print(f"\n{category}:")
    found_cols = [col for col in columns if col in df.columns]
    if found_cols:
        for col in found_cols[:5]:  # Show first 5
            missing_pct = missing_summary.loc[col, 'Missing_Percentage'] if col in missing_summary.index else 0
            print(f"  ✅ {col}: {missing_pct:.1f}% missing")
        if len(found_cols) > 5:
            print(f"  ... and {len(found_cols)-5} more columns")
    else:
        print(f"  ❌ None of these columns found")

# ============================================
# PART 4: VISUALIZATION OF MISSING DATA
# ============================================

# Create visualization directory
import os
os.makedirs('missing_data_plots', exist_ok=True)

# Plot 1: Missing data distribution
plt.figure(figsize=(16, 10))
plt.hist(missing_summary['Missing_Percentage'], bins=50, color='steelblue', edgecolor='black')
plt.xlabel('Missing Percentage (%)', fontsize=12)
plt.ylabel('Number of Columns', fontsize=12)
plt.title('Distribution of Missing Data Across Columns', fontsize=14, fontweight='bold')
plt.axvline(x=50, color='red', linestyle='--', label='50% threshold')
plt.legend()
plt.savefig('missing_data_plots/missing_data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 2: Heatmap of missing data (top 100 columns)
plt.figure(figsize=(20, 15))
top_100_missing = missing_summary.tail(100).index
missing_binary = df[top_100_missing].isnull().astype(int)
sns.heatmap(missing_binary, cbar=True, cmap='viridis', yticklabels=False)
plt.title('Missing Data Heatmap (Top 100 Columns with Most Missing)', fontsize=14, fontweight='bold')
plt.xlabel('Columns', fontsize=12)
plt.tight_layout()
plt.savefig('missing_data_plots/missing_data_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 3: Missingness by category
categories_missing = {}
for category, columns in key_columns.items():
    found_cols = [col for col in columns if col in df.columns]
    if found_cols:
        avg_missing = missing_summary.loc[found_cols, 'Missing_Percentage'].mean()
        categories_missing[category] = avg_missing

plt.figure(figsize=(12, 6))
plt.barh(list(categories_missing.keys()), list(categories_missing.values()), color='coral')
plt.xlabel('Average Missing Percentage (%)', fontsize=12)
plt.title('Average Missing Data by Clinical Category', fontsize=14, fontweight='bold')
plt.xlim(0, 100)
plt.tight_layout()
plt.savefig('missing_data_plots/missing_by_category.png', dpi=150, bbox_inches='tight')
plt.show()


## Feature Distributions

[Plot the distribution of various features and target variables. Comment on the skewness, outliers, or any other observations.]


In [ ]:
# Example: Plotting histograms of all numerical features
df.hist(figsize=(12, 12))
plt.show()


## Possible Biases

[Investigate the dataset for any biases that could affect the model’s performance and fairness (e.g., class imbalance, historical biases).]


In [ ]:
# Example: Checking for class imbalance in a classification problem
# sns.countplot(x='target_variable', data=df)

# Your code to investigate possible biases goes here


## Correlations

[Explore correlations between features and the target variable, as well as among features themselves.]


In [ ]:
# Example: Plotting a heatmap to show feature correlations
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True)
plt.show()
